In [ ]:
%load_ext autoreload
%autoreload 2

### **Paso 1: Generación de datos**

In [ ]:
# Generación de los datos sintéticos
from data_generation.data_config import DATA_CONFIG
from data_generation.simulator import DataSimulator

In [ ]:
# Paso 1: generar los datos
panel_simulator = DataSimulator(DATA_CONFIG)
panel = panel_simulator.simulate()
panel_simulator.export_panel_and_config()

In [ ]:
# Paso 1.1: análisis exploratorio de los datos
print(panel.head())
print()
print(panel.info())
print()
print(list(panel.columns))

### **Paso 2: Split en train y test**

In [ ]:
# Divisón de IDs en train y test
from splits.split_generator import SplitGenerator

In [ ]:
# Paso 2: generar el split train/test. El split_generator ya está configurado
# para que al train vayan todos los tratados y un porcentaje de los nini, y al
# test vayan todos los controles y el resto de los nini
split_generator = SplitGenerator(panel, train_nini_ratio=0.5, seed=13)
train_ids, test_ids = split_generator.generate()

In [ ]:
# Paso 2.1: revisar que el split se hizo correctamente
split = split_generator.split

train = split['train']
test = split['test']

treated = train['T']
nini_train = train['NiNi']

control = test['C']
nini_test = test['NiNi']

### **Paso 3: Conversión de datos a tensores de PyTorch y formamos `Datasets`**

In [ ]:
from connectors.lstm import LSTMConnector

In [ ]:
FEATURE_COLS = [
    'antiguedad',
    'ratio_formalidad',
    'empleados',
    'salario_promedio'
]

In [ ]:
lstm_connector = LSTMConnector(
    panel=panel,
    split=split_generator.split,
    feature_cols=FEATURE_COLS
)

train_dataset, test_dataset = lstm_connector.convert(fit_scaler=False)

In [ ]:
print(len(train_dataset))
print(len(test_dataset))

print(train_dataset[0])
print(test_dataset[0])

In [ ]:
for (*X, y) in train_dataset[:10]:
    temporal = X[1]
    print(X[0], temporal.shape)   # Vemos que son secuencias de largo variable

# for (*X, y) in test_dataset[:10]:
#     temporal = X[0]
#     print(temporal.shape)

### **Paso 4: Creamos los DataLoaders**

Hacer esto no es tan directo porque tenemos secuencias de largo variable.
Existen dos alternativas:
1. Usar `padding` y `packing`.
2. Organizar los lotes de tal manera que cada lote tenga secuencias de la misma
longitud.

Por ahora, vamos con la opción 1.

Algunas referencias:
- [Discuss PyTorch - Different length sequences as batch input to LSTM](https://discuss.pytorch.org/t/different-length-sequences-as-batch-input-to-lstm/66119)
- [Discuss PyTorch - Understanding pack_padded_sequence and pad_packed_sequence](https://discuss.pytorch.org/t/understanding-pack-padded-sequence-and-pad-packed-sequence/4099/15)

In [ ]:
from torch.utils.data import DataLoader
from connectors.lstm import PanelSequenceDataset

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=PanelSequenceDataset.collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=PanelSequenceDataset.collate_fn
)

In [ ]:
# Veamos un batch de ejemplo
batch = next(iter(train_loader))

print(f"IDs: {batch[0].shape}\n  {batch[0]}\n")   # (batch_size,)
print(f"Secuencias: {batch[1].shape}\n")          # (batch_size, T_max, F)
print(f"Largos originales:\n\t{batch[2]}\n")      # (batch_size,)
print(f"IDs de cohortes: \n\t{batch[3]}\n")       # (batch_size,)
print(f"Labels: \n\t{batch[4]}")                  # (batch_size,)

### **Paso 5: Definir la función de pérdida**

In [ ]:
import torch
import torch.nn as nn

from data_generation.panel_schema import Col

n_nini    = len(split_generator.split['train']['NiNi'])
n_treated = len(split_generator.split['train']['T'])
n_cohorts = len(panel.loc[panel[Col.TRATADO_EN_T] == 1, Col.T].unique())

print(f"n_nini: {n_nini}, n_treated: {n_treated}, n_cohorts: {n_cohorts}")

pos_weight = torch.tensor(n_nini * n_cohorts / n_treated, dtype=torch.float32)

print(f"Pos weight: {pos_weight}")

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

### **Paso 6: Búsqueda de hiperparámetros con Optuna**

En primer lugar, los hiperparámetros son de dos tipos:
- Hiperparámetros del modelo: número de capas, número de neuronas por capa, función de activación, etc. **Dependen de la arquitectura del modelo.**
- Hiperparámetros del entrenamiento: tasa de aprendizaje, número de épocas, tamaño del lote, etc. **Son comunes a cualquier modelo.**

Para empezar, estos son sobre los que vamos a hacer la búsqueda de hiperparámetros:
- De entrenamiento:
    - Tasa de aprendizaje.
- De arquitectura:
    - Para LSTM:
        - Número de capas LSTM.
        - Número de neuronas por capa LSTM.
        - Dropout entre capas LSTM.

In [ ]:
import optuna

In [ ]:
# Vamos a establecer los rangos de búsqueda en para cada hiperparámetro
# de antemano
DROPOUTS = [0.3, 0.5, 0.7]
HIDDEN_SIZES = [32, 64, 128, 256]
LEARNING_RATES = [1e-4, 1e-3, 1e-2]
LSTM_LAYERS = [2, 3, 4]

# Y a algunos otros los dejamos fijos
N_EPOCHS = 10
OPTIMIZER = "Adam"

In [ ]:
from models.lstm_classifier import LSTMClassifier
from training.trainer import Trainer

def objective(trial):
    # Hiperparámetros de entrenamiento
    lr = trial.suggest_categorical("learning_rate", LEARNING_RATES)

    # Hiperparámetros del modelo
    dropout = trial.suggest_categorical("dropout", DROPOUTS)
    lstm_hidden_size = trial.suggest_categorical("lstm_hidden_size", HIDDEN_SIZES)
    lstm_num_layers = trial.suggest_categorical("lstm_num_layers", LSTM_LAYERS)

    # Creamos el modelo con los hiperparámetros sugeridos
    model = LSTMClassifier(
        n_features=len(FEATURE_COLS),
        n_cohorts=n_cohorts,
        lstm_hidden_size=lstm_hidden_size,
        lstm_num_layers=lstm_num_layers,
        dropout=dropout
    )

    # Creamos el optimizador con la tasa de aprendizaje sugerida
    optimizer = getattr(torch.optim, OPTIMIZER)(model.parameters(), lr=lr)

    # Entrenamos el modelo con todos los parámetros sugeridos.
    # Acá se usa solamente el conjunto de entrenamiento
    trainer = Trainer(
        model=model,
        optimizer=optimizer,
        criterion=loss_fn,
        device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    )

    trainer.fit(train_loader, test_loader, n_epochs=N_EPOCHS)

    # Evaluamos la accuracy en el conjunto de test
    accuracy = trainer.accuracy(test_loader)

    return accuracy

In [ ]:
OPTUNA_STORAGE = "sqlite:///db.sqlite3"

In [ ]:
import time

timestamp = time.strftime("%d%m%Y-%H%M%S")
study_name = f"study_{timestamp}"

In [ ]:
study = optuna.create_study(
    direction="maximize",
    storage=OPTUNA_STORAGE,
    study_name=study_name,
    load_if_exists=True
)

In [ ]:
study.optimize(objective, n_trials=5)

In [ ]:
print(study.best_trial.value, study.best_trial.params)

### **Paso 7: Instanciar y entrenar el modelo con los mejores hiperparámetros**

In [ ]:
lstm_hidden_size = study.best_trial.params['lstm_hidden_size']
lstm_num_layers = study.best_trial.params['lstm_num_layers']
dropout = study.best_trial.params['dropout']

model = LSTMClassifier(
    n_features=len(FEATURE_COLS),
    lstm_hidden_size=lstm_hidden_size,
    lstm_num_layers=lstm_num_layers,
    n_cohorts=n_cohorts,
    dropout=dropout
)

lr = study.best_trial.params['learning_rate']

optimizer = torch.optim.Adam(model.parameters(), lr=lr)

### **Paso 8: Entrenamiento**

In [ ]:
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=loss_fn,
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
)

In [ ]:
trainer.fit(train_loader, test_loader, n_epochs=10)

In [ ]:
accuracy = trainer.accuracy(test_loader)
print(f"Test accuracy: {accuracy}")